# Workflow: From FCHK to electron density on a grid

This notebook demonstrates a simple workflow using [IOData](https://github.com/theochem/iodata), [Grid](https://github.com/theochem/grid), and [GBasis](https://github.com/theochem/gbasis): load a wavefunction from a Gaussian `.fchk` file, build a molecular grid, and evaluate the electron density on that grid.

> **If you are using Google Colab**, run the next cell first to install the packages and download the example file.

In [3]:
# Install packages in Google Colab. Skip this cell if you already have the packages and data.
! pip install git+https://github.com/theochem/iodata.git
! pip install git+https://github.com/theochem/grid.git
! pip install git+https://github.com/theochem/gbasis.git

import os
from urllib.request import urlretrieve

fpath = "data/"
if not os.path.exists(fpath):
    os.makedirs(fpath, exist_ok=True)
urlretrieve(
    "https://raw.githubusercontent.com/theochem/horton3/master/notebooks/data/h2o_sto3g.fchk",
    os.path.join(fpath, "h2o_sto3g.fchk")
)

  Cloning https://github.com/theochem/iodata.git to /private/var/folders/wy/51q5nkq91nx7p_h2z4vpn_dr0000gn/T/pip-req-build-7052guvo
  Running command git clone --filter=blob:none --quiet https://github.com/theochem/iodata.git /private/var/folders/wy/51q5nkq91nx7p_h2z4vpn_dr0000gn/T/pip-req-build-7052guvo
  Resolved https://github.com/theochem/iodata.git to commit e6a606e985a4a0010d64982c526efa1b95ac09b6
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
  Cloning https://github.com/theochem/grid.git to /private/var/folders/wy/51q5nkq91nx7p_h2z4vpn_dr0000gn/T/pip-req-build-q4xptdc9
  Running command git clone --filter=blob:none --quiet https://github.com/theochem/grid.git /private/var/folders/wy/51q5nkq91nx7p_h2z4vpn_dr0000gn/T/pip-req-build-q4xptdc9
done
  Preparing metadata (pyproject.toml) ..

('data/h2o_sto3g.fchk', <http.client.HTTPMessage at 0x10b2f9450>)

## 1. Load the wavefunction with IOData

We load a formatted checkpoint (`.fchk`) file produced by Gaussian. IOData parses geometry, basis set, and molecular orbital data.

In [4]:
from iodata import load_one

mol = load_one("data/h2o_sto3g.fchk")

print("Number of atoms:", mol.natom)
print("Atomic numbers:", mol.atnums)
print("Atomic coordinates (bohr):")
print(mol.atcoords)
print("Number of basis functions:", mol.obasis.nbasis)

Number of atoms: 3
Atomic numbers: [8 1 1]
Atomic coordinates (bohr):
[[-4.44734101  3.39697999  0.        ]
 [-2.58401495  3.55136194  0.        ]
 [-4.92380519  5.2049622   0.        ]]
Number of basis functions: 7


## 2. Build a molecular grid with Grid

We construct an atom-centered grid with Becke partitioning. This grid will be used to evaluate the density numerically.

In [ ]:
from grid.becke import BeckeWeights
from grid.molgrid import MolGrid
from grid.onedgrid import GaussChebyshev
from grid.rtransform import BeckeRTransform

oned = GaussChebyshev(100)
rgrid = BeckeRTransform(1e-4, 1.5).transform_1d_grid(oned)
grid = MolGrid.from_size(mol.atnums, mol.atcoords, 110, rgrid, BeckeWeights())

print("Grid points shape:", grid.points.shape)
print("Grid weights shape:", grid.weights.shape)

## 3. Evaluate the electron density with GBasis

We get the one-particle density matrix from the loaded wavefunction, then use GBasis to evaluate the density at each grid point.

In [ ]:
import numpy as np
from gbasis.wrappers import from_iodata
from gbasis.evals.density import evaluate_density

# One-particle density matrix from the SCF (or post-SCF) solution
one_rdm = mol.one_rdms.get("post_scf", mol.one_rdms.get("scf"))
if one_rdm is None:
    if mol.mo is None:
        raise ValueError(
            "The input file lacks wavefunction data with which "
            "the density can be computed."
        )
    coeffs, occs = mol.mo.coeffs, mol.mo.occs
    one_rdm = np.dot(coeffs * occs, coeffs.T)

basis = from_iodata(mol)
density = evaluate_density(one_rdm, basis, grid.points)

print("Density shape:", density.shape)
# Integrate density * weights to get electron count (should be ~10 for H2O)
n_electrons = np.sum(grid.weights * density)
print("Integrated electron count: {:.4f}".format(n_electrons))